# 🦟 JORINOVA NEXUS — Malaria Parasite Detector (one-click training)

Trains a **YOLOv8** detector on **Roboflow** + **NIH** malaria microscopy data and produces
`malaria.pt` for the app's vision service.

**Before you start:** Runtime → **Change runtime type** → Hardware accelerator → **T4 GPU**.

**No Roboflow key needed** — step 4 **Option A** uses a public NIH dataset (BBBC041).
Roboflow is offered as an optional alternative. Run each cell top-to-bottom (Shift+Enter).


## 1. Check the GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'


## 2. Install training dependencies


In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 3. Get the project code
Pulls `ml/malaria/` (class map + scripts) and the folder the weights go into.
If your GitHub repo is **private**, paste a token (repo scope) when asked; otherwise press Enter.


In [ ]:
import os
from getpass import getpass
REPO = 'https://github.com/dujoely1/JORINOVA-NEXUS.git'
tok = getpass('GitHub token (press Enter if the repo is public): ').strip()
url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
os.system(f'rm -rf /content/nexus && git clone --depth 1 {url} /content/nexus')
ROOT = '/content/nexus/ml/malaria' if os.path.exists('/content/nexus/ml/malaria') else '/content'
os.chdir(ROOT)
print('working dir:', os.getcwd())
print('repo cloned:', os.path.exists('/content/nexus/ml/malaria'))


## 4. Get the training data — run **Option A** *or* Option B
**Option A needs NO Roboflow key** — use it and skip Option B.


### ✅ Option A — public NIH BBBC041 (no key needed)
Downloads the public malaria bounding-box set and converts it to YOLO format.
Run this one cell and jump to step 5.


In [ ]:
import os, json, zipfile, urllib.request, shutil
from pathlib import Path
os.makedirs('/content/data', exist_ok=True)
zp = '/content/data/malaria.zip'
if not os.path.exists(zp):
    print('downloading BBBC041 (a few hundred MB, ~1-2 min)...')
    urllib.request.urlretrieve('https://data.broadinstitute.org/bbbc/BBBC041/malaria.zip', zp)
with zipfile.ZipFile(zp) as z: z.extractall('/content/data/bbbc041')
CAT = {'red blood cell':0,'leukocyte':1,'ring':2,'trophozoite':3,'schizont':4,'gametocyte':5,'difficult':6}
NAMES = ['red_blood_cell','leukocyte','ring','trophozoite','schizont','gametocyte','difficult']
root = Path('/content/data/bbbc041')
def find(name):
    h = list(root.rglob(name)); return h[0] if h else None
def convert(jp, split):
    io = Path('/content/data/bbbc041_yolo/images/'+split); lo = Path('/content/data/bbbc041_yolo/labels/'+split)
    io.mkdir(parents=True, exist_ok=True); lo.mkdir(parents=True, exist_ok=True)
    n = 0
    for e in json.loads(jp.read_text()):
        im = e.get('image', {}); sh = im.get('shape', {}); H = sh.get('r'); W = sh.get('c'); pn = im.get('pathname','')
        if not (H and W and pn): continue
        src = find(Path(pn).name)
        if not src: continue
        lines = []
        for o in e.get('objects', []):
            cid = CAT.get(str(o.get('category','')).lower())
            if cid is None: continue
            b = o.get('bounding_box', {}); mn = b.get('minimum', {}); mx = b.get('maximum', {})
            if None in (mn.get('r'), mn.get('c'), mx.get('r'), mx.get('c')): continue
            xc = ((mn['c']+mx['c'])/2)/W; yc = ((mn['r']+mx['r'])/2)/H
            w = abs(mx['c']-mn['c'])/W; h = abs(mx['r']-mn['r'])/H
            lines.append(f'{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        shutil.copy(src, io/src.name); (lo/f'{src.stem}.txt').write_text(chr(10).join(lines)); n += 1
    return n
tj = find('training.json') or find('train.json'); vj = find('test.json')
nt = convert(tj, 'train'); nv = convert(vj, 'val') if vj else 0
if nv == 0:  # no test split -> carve 15%
    imgs = sorted(Path('/content/data/bbbc041_yolo/images/train').glob('*'))
    for p in imgs[:max(1, len(imgs)//7)]:
        Path('/content/data/bbbc041_yolo/images/val').mkdir(parents=True, exist_ok=True)
        Path('/content/data/bbbc041_yolo/labels/val').mkdir(parents=True, exist_ok=True)
        shutil.move(str(p), '/content/data/bbbc041_yolo/images/val/'+p.name)
        lb = Path('/content/data/bbbc041_yolo/labels/train/'+p.stem+'.txt')
        if lb.exists(): shutil.move(str(lb), '/content/data/bbbc041_yolo/labels/val/'+lb.name)
        nv += 1; nt -= 1
DATA_YAML = '/content/data/bbbc041_yolo/data.yaml'
open(DATA_YAML,'w').write(f'path: /content/data/bbbc041_yolo\ntrain: images/train\nval: images/val\nnc: {len(NAMES)}\nnames: {NAMES}\n')
print(f'✓ {nt} train / {nv} val images  ->  {DATA_YAML}')


### Option B — a Roboflow project (only if you skipped Option A)
Needs your API key (app.roboflow.com → Settings → Roboflow API). On **Roboflow Universe**
search *malaria*, open a bounding-box project, **Download → YOLOv8**, copy the slugs below.


In [ ]:
# Skip if you ran Option A. Otherwise fill these in and run:
# from getpass import getpass
# os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
# WORKSPACE = 'workspace-slug'; PROJECT = 'malaria-project-slug'; VERSION = 1
# from roboflow import Roboflow
# rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8', location='/content/data/roboflow')
# DATA_YAML = os.path.join(ds.location, 'data.yaml'); print('DATA_YAML =', DATA_YAML)


## 6. (Optional) NIH cell-image set
Public ~337 MB Parasitized/Uninfected set — used by the classifier in the last (optional) cell.
Skip if you only want the detector.


In [ ]:
# Uncomment to fetch the NIH classification set:
# !wget -q https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip -O /content/cell_images.zip
# !unzip -q /content/cell_images.zip -d /content/data/nih_cells
# print('NIH images:', len(os.listdir('/content/data/nih_cells/cell_images')))


## 7. Train the YOLOv8 detector
~80 epochs on a T4 takes roughly 30–90 min depending on dataset size. Augmentation is tuned
for stained microscopy (full rotation/flip). Lower `EPOCHS` for a quick smoke test.


In [ ]:
EPOCHS = 80
from ultralytics import YOLO
model = YOLO('yolov8s.pt')  # yolov8n.pt = faster, yolov8m.pt = more accurate
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, name='malaria',
                      patience=20, plots=True,
                      hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, degrees=180,
                      fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
metrics = model.val()
print(f'mAP50 = {metrics.box.map50:.3f}   mAP50-95 = {metrics.box.map:.3f}')


## 8. Preview predictions on a few validation images


In [ ]:
import glob
from IPython.display import Image, display
val_imgs = glob.glob(os.path.join(ds.location, 'valid', 'images', '*'))[:3] or \
           glob.glob(os.path.join(ds.location, 'test', 'images', '*'))[:3]
if val_imgs:
    model.predict(val_imgs, save=True, conf=0.25, name='malaria_preview')
    for p in glob.glob('/content/runs/detect/malaria_preview*/**/*.jpg', recursive=True)[:3]:
        display(Image(p, width=460))
else:
    print('no val/test images found to preview')


## 9. Export `malaria.pt` and download it
Copies the best weights into the app's model folder and downloads them to your computer.


In [ ]:
import shutil, glob, os
best = sorted(glob.glob('/content/runs/detect/malaria*/weights/best.pt'), key=os.path.getmtime)[-1]
os.makedirs('/content/nexus/backend/models/malaria', exist_ok=True)
dest = '/content/nexus/backend/models/malaria/malaria.pt'
shutil.copy(best, dest)
print('saved to app path:', dest)
# also export ONNX (portable)
try:
    model.export(format='onnx', opset=12)
except Exception as e:
    print('onnx export skipped:', e)
from google.colab import files
files.download(best)  # downloads best.pt to your machine


## 10. Put it in the app
1. Move the downloaded `best.pt` to **`backend/models/malaria/malaria.pt`** in your repo.
2. `git add backend/models/malaria/malaria.pt && git commit -m 'malaria weights' && git push` (merge to `main`).
3. On Render, add `ANTHROPIC_API_KEY` (for the Claude-vision fallback) and add `ultralytics` to the backend requirements so the detector loads.

The vision service then runs **your detector first** on blood-smear / parasitology images (stage counts + parasitaemia), and Claude vision as the narrative/fallback.


## 11. (Optional) NIH parasitized-vs-uninfected classifier
A fast screening head. Run cell 6 first to download the NIH set.


In [ ]:
# Optional — needs the NIH cell_images from cell 6.
import torch, os
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
NIH = '/content/data/nih_cells/cell_images'
if os.path.exists(NIH):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tf = transforms.Compose([transforms.Resize((128,128)), transforms.ToTensor(),
                             transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    full = datasets.ImageFolder(NIH, transform=tf); classes = full.classes
    nval = int(len(full)*0.15)
    tr, va = random_split(full, [len(full)-nval, nval], generator=torch.Generator().manual_seed(42))
    tl = DataLoader(tr, batch_size=64, shuffle=True, num_workers=2); vl = DataLoader(va, batch_size=64, num_workers=2)
    net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT); net.fc = nn.Linear(net.fc.in_features, len(classes)); net = net.to(dev)
    opt = torch.optim.Adam(net.parameters(), 1e-4); lossf = nn.CrossEntropyLoss()
    for ep in range(6):
        net.train()
        for x,y in tl:
            x,y = x.to(dev), y.to(dev); opt.zero_grad(); lossf(net(x), y).backward(); opt.step()
        net.eval(); c=t=0
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(dev),y.to(dev); c+=(net(x).argmax(1)==y).sum().item(); t+=y.size(0)
        print(f'epoch {ep+1} val_acc={c/max(t,1):.4f}')
    os.makedirs('/content/nexus/backend/models/malaria', exist_ok=True)
    torch.save({'state_dict': net.state_dict(), 'classes': classes, 'arch':'resnet18','img_size':128},
               '/content/nexus/backend/models/malaria/malaria_cls.pt')
    print('saved classifier; classes =', classes)
else:
    print('NIH set not found — run cell 6 first')
